# 06 - Memoria e Estado
> Short-term, long-term e memoria externa para agentes

**Modulo:** `04_agentes` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import json, os, datetime

class MemoriaAgente:
    def __init__(self, arquivo='memoria.json', max_hist=8):
        self.arquivo=arquivo; self.max_hist=max_hist
        self.hist=[]
        self.fatos=json.load(open(arquivo)) if os.path.exists(arquivo) else []

    def lembrar(self, fato):
        self.fatos.append({'fato':fato,'quando':datetime.datetime.now().isoformat()})
        json.dump(self.fatos, open(self.arquivo,'w'), ensure_ascii=False)

    def chat(self, msg):
        ctx='\n'.join(f'- {f["fato"]}' for f in self.fatos[-5:]) or 'Nenhuma memoria.'
        self.hist.append({'role':'user','content':msg})
        if len(self.hist)>self.max_hist: self.hist=self.hist[-self.max_hist:]
        r=client.messages.create(model=HAIKU,max_tokens=256,
            system=f'Voce tem memoria persistente.\nFatos:\n{ctx}',
            messages=self.hist)
        resp=r.content[0].text
        self.hist.append({'role':'assistant','content':resp})
        if any(p in msg.lower() for p in ['meu nome','eu sou','eu trabalho']): self.lembrar(msg)
        return resp

m=MemoriaAgente()
for c in ['Meu nome e Carlos e sou dev Python.','Trabalho numa fintech em POA.','Qual meu nome?']:
    print(f'Voce: {c}'); print(f'Bot: {m.chat(c)}\n')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
